Fixed double-slash bug in stage file download path
*Co-authored with CoCo*

# Lab: GenAI Cortex AI Functions Part 3

📚 In this lab you will learn and practice the following:

❄️ Use AI_COMPLETE with vision-capable models to analyze and describe images

❄️ Classify emotions and content in photographs using AI_CLASSIFY

❄️ Analyze video files and extract structured metadata using AI_COMPLETE with JSON schemas

❄️ Transcribe audio recordings with AI_TRANSCRIBE

❄️ Use AI_FILTER and AI-powered joins to connect data by meaning rather than keys

❄️ Combine unstructured data (images, video, audio) with structured data using SQL

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

In this lab, you will explore multimodal analytics in Snowflake - combining unstructured data like images, video, and audio with structured data using Cortex AI functions, all with a few lines of SQL.

You will classify emotions in photographs, extract metadata from video, transcribe audio recordings, and use AI-powered filtering and joins to connect data by meaning rather than by keys.

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: LLM Functions Part 3';
SHOW PARAMETERS LIKE 'query_tag' in session 
->> SELECT "value" AS query_tag FROM $1;

## AI_COMPLETE For Images

AI_COMPLETE supports vision-capable models that can analyze images, generate descriptions, compare multiple images, and answer questions about visual content - directly in SQL using `PROMPT()` and `TO_FILE()`.

### Download multimodal files for this lab

The following cell downloads sample images, audio, and video files from the stage to your local workspace. These files are used throughout the lab for image analysis, audio transcription, and video metadata extraction.

In [ ]:
import os
os.makedirs('../images', exist_ok=True)
os.makedirs('../audio', exist_ok=True)
os.makedirs('../video', exist_ok=True)

stage = f'@{user}_genai_db.resources.genai2day/source_files'

# Images
image_files = ['expressions.jpeg', 'fridge1.jpeg', 'fridge2.jpeg', 'uneven.jpg']
for f in image_files:
    session.file.get(f'{stage}/{f}', '../images/')

# Audio
audio_files = ['garrandarra_merlot_review.mp3']
for f in audio_files:
    session.file.get(f'{stage}/{f}', '../audio/')

# Video (from shared genai_db stage)
video_stage = '@genai_db.resources.genai2day/source_files'
video_files = ['hiking_shop_gear.mp4']
for f in video_files:
    session.file.get(f'{video_stage}/{f}', '../video/')

print('Multimodal files downloaded: ../images/, ../audio/, ../video/')

In [ ]:
import os
os.makedirs('../images', exist_ok=True)
os.makedirs('../audio', exist_ok=True)
os.makedirs('../video', exist_ok=True)

stage = f'@{user}_genai_db.resources.beetle_files'

# Images
# image_files = ['expressions.jpeg', 'fridge1.jpeg', 'fridge2.jpeg', 'uneven.jpg']

image_files = ['Gathering.png', 'ibm_img.png', 'IMG_7962.jpg', 'IMG_7972.jpg']
for f in image_files:
    session.file.get(f'{stage}/{f}', '../images/')

# Audio
# audio_files = ['garrandarra_merlot_review.mp3']
# for f in audio_files:
#     session.file.get(f'{stage}/{f}', '../audio/')

# # Video (from shared genai_db stage)
# video_stage = '@genai_db.resources.genai2day/source_files'
# video_files = ['hiking_shop_gear.mp4']
# for f in video_files:
#     session.file.get(f'{video_stage}/{f}', '../video/')

print('Multimodal files downloaded: ../images/')

### Image analysis - classify emotions.

Run AI_COMPLETE with claude-4-haiku against a photograph to classify the emotions expressed in it.

In [ ]:
from IPython.display import display, Image
# display(Image(filename='../images/expressions.jpeg', width=400))
display(Image(filename='../images/Gathering.png', width=400))
display(Image(filename='../images/ibm_img.png', width=400))
display(Image(filename='../images/IMG_7972.jpg', width=400))
display(Image(filename='../images/IMG_7962.jpg', width=400))

In [ ]:
%%sql -r Image_analysis_emotions_sql
SELECT AI_COMPLETE('claude-haiku-4-5',
  PROMPT('Classify the emotions expressed in the input image {0}',
    -- TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'expressions.jpeg')
    -- TO_FILE('@{{user}}_genai_db.resources.beetle_files', 'Gathering.png')
    -- TO_FILE('@{{user}}_genai_db.resources.beetle_files', 'ibm_img.png')
    -- TO_FILE('@{{user}}_genai_db.resources.beetle_files', 'IMG_7972.jpg')
    TO_FILE('@{{user}}_genai_db.resources.beetle_files', 'IMG_7962.jpg')
  )
) AS response;


In [ ]:

df = Image_analysis_emotions_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["RESPONSE"].iloc[0]

# Replace the literal '\\n' string with a true newline character '\n'
corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

### Compare multiple images.

Pass two images into a single AI_COMPLETE call using `{0}` and `{1}` placeholders in the prompt. The model compares the contents and describes the differences.

![Fridge 1](../images/fridge1.jpeg) ![Fridge 2](../images/fridge2.jpeg)

In [ ]:
from IPython.display import display, Image, HTML

display(HTML("<div style='display:flex; gap:16px;'>"))
display(Image(filename='../images/fridge1.jpeg', width=300))
display(Image(filename='../images/fridge2.jpeg', width=300))
display(HTML("</div>"))

In [ ]:
%%sql -r Comparing_multiple_images_sql
SELECT AI_COMPLETE('claude-haiku-4-5',
  PROMPT('Compare this image {0} to this image {1} and describe the differences.',
    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'fridge1.jpeg'),
    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files', 'fridge2.jpeg')
  )
) response;


In [ ]:
# Format output from the previous cell
df = Comparing_multiple_images_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["RESPONSE"].iloc[0]

# Replace the literal '\\n' string with a true newline character '\n'
corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

### Detailed image analysis and risk assessment.

Analyze a hiking trail photograph to identify the location and flag potential safety risks for trail management.

In [ ]:
%%sql -r Generating_detailed_image_descriptions_sql
SELECT AI_COMPLETE('claude-haiku-4-5',
  PROMPT('The following image was taken by our Travelbug hiking guide Malia for further analysis. Can you identify where it was taken? Identify potential risks to those on this track {0}',
    TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files','uneven.jpg') 
  ) 
) response;

In [ ]:

df = Generating_detailed_image_descriptions_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["RESPONSE"].iloc[0]

# Replace the literal '\\n' string with a true newline character '\n'
corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

### Video analysis - extract structured metadata.

Use AI_COMPLETE with `gemini-3.1-pro` to analyze a video file and return structured JSON metadata. The video must be stored in a Snowflake stage as a FILE column - create a table first, then query it.

📌 **Note:** Video analysis with AI_COMPLETE is in **Public Preview**. The file is passed as a third positional argument (not inside PROMPT()) and requires a vision-capable model such as `gemini-3.1-pro`.

In [ ]:
import base64
from IPython.display import display, HTML

video_path = '../video/hiking_shop_gear.mp4'
with open(video_path, 'rb') as f:
    video_b64 = base64.b64encode(f.read()).decode('utf-8')

html = (
    '<h4>Preview: hiking_shop_gear.mp4</h4>'
    '<video controls width="640">'
    f'<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">'
    'Your browser does not support video playback.'
    '</video>'
)
display(HTML(html))

In [ ]:
%%sql -r Video_analysis_create_table_sql
-- Step 1: Create a table with the video file as a FILE column
-- (Video analysis requires a FILE column - it cannot be called inline)
CREATE OR REPLACE TEMPORARY TABLE {{user}}_genai_db.resources.travelbug_video AS
SELECT TO_FILE('@genai_db.resources.genai2day', 'source_files/hiking_shop_gear.mp4') AS video_file;

In [ ]:
%%sql -r Video_analysis_extract_metadata_sql
-- Step 2: Analyze the video and extract structured metadata
SELECT AI_COMPLETE(
    'gemini-3.1-pro',
    'Analyze this video of hiking gear on display and extract the following information as JSON. Only include details you can clearly observe in the video.',
    video_file,
    {},
    {
        'type': 'json',
        'schema': {
            'type': 'object',
            'properties': {
                'setting':       {'type': 'string', 'description': 'Where the video was filmed e.g. indoor retail store, warehouse'},
                'products': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'category':     {'type': 'string', 'description': 'Type of gear e.g. hiking poles, gloves, backpacks'},
                            'colors':       {'type': 'array', 'items': {'type': 'string'}, 'description': 'Colors visible for this product'},
                            'quantity':     {'type': 'string', 'description': 'Approximate number of items visible e.g. several, a rack of 10'},
                            'display_type': {'type': 'string', 'description': 'How products are displayed e.g. hanging on rack, on shelf'}
                        }
                    }
                },
                'summary':       {'type': 'string', 'description': 'Brief description of what the video shows'}
            },
            'required': ['setting', 'products', 'summary']
        }
    }
) AS video_analysis
FROM {{user}}_genai_db.resources.travelbug_video;

In [ ]:
import json
from IPython.display import display, HTML

df = Video_analysis_extract_metadata_sql.to_pandas()
result = json.loads(df["VIDEO_ANALYSIS"].iloc[0])

html = "<div style='font-family:sans-serif; padding:16px; border:1px solid #ddd; border-radius:8px;'>"
html += "<h3>Hiking Gear Video Analysis</h3>"
html += f"<p><strong>Setting:</strong> {result.get('setting', '')}</p>"
html += f"<p><strong>Summary:</strong> {result.get('summary', '')}</p>"
html += "<h4>Products Identified:</h4><table style='border-collapse:collapse; width:100%;'>"
html += "<tr style='background:#f0f0f0;'><th style='padding:8px; text-align:left;'>Category</th><th style='padding:8px; text-align:left;'>Colors</th><th style='padding:8px; text-align:left;'>Quantity</th><th style='padding:8px; text-align:left;'>Display</th></tr>"
for p in result.get('products', []):
    colors = ', '.join(p.get('colors', []))
    html += f"<tr><td style='padding:8px; border-top:1px solid #ddd;'>{p.get('category','')}</td>"
    html += f"<td style='padding:8px; border-top:1px solid #ddd;'>{colors}</td>"
    html += f"<td style='padding:8px; border-top:1px solid #ddd;'>{p.get('quantity','')}</td>"
    html += f"<td style='padding:8px; border-top:1px solid #ddd;'>{p.get('display_type','')}</td></tr>"
html += "</table></div>"
display(HTML(html))

In [ ]:
%%sql -r video_categories
-- Extract and compare product categories from hiking gear video metadata
SELECT 
    video_analysis:"setting"::STRING AS setting,
    video_analysis:"summary"::STRING AS summary,
    f.value:"category"::STRING AS category,
    f.value:"colors"::ARRAY AS colors,
    f.value:"quantity"::STRING AS quantity,
    f.value:"display_type"::STRING AS display_type,
    f.index + 1 AS product_rank
FROM (
    SELECT AI_COMPLETE(
        'gemini-3.1-pro',
        'Analyze this video of hiking gear on display and extract the following information as JSON. Only include details you can clearly observe in the video.',
        video_file,
        {},
        {
            'type': 'json',
            'schema': {
                'type': 'object',
                'properties': {
                    'setting':       {'type': 'string', 'description': 'Where the video was filmed e.g. indoor retail store, warehouse'},
                    'products': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'properties': {
                                'category':     {'type': 'string', 'description': 'Type of gear e.g. hiking poles, gloves, backpacks'},
                                'colors':       {'type': 'array', 'items': {'type': 'string'}, 'description': 'Colors visible for this product'},
                                'quantity':     {'type': 'string', 'description': 'Approximate number of items visible e.g. several, a rack of 10'},
                                'display_type': {'type': 'string', 'description': 'How products are displayed e.g. hanging on rack, on shelf'}
                            }
                        }
                    },
                    'summary':       {'type': 'string', 'description': 'Brief description of what the video shows'}
                },
                'required': ['setting', 'products', 'summary']
            }
        }
    ) AS video_analysis
    FROM {{user}}_genai_db.resources.travelbug_video
) v,
LATERAL FLATTEN(input => v.video_analysis:"products") f
ORDER BY f.index;

## AI_TRANSCRIBE

Converts speech from audio or video files to text. Supports word-level or speaker-level timestamps. Maximum file duration: 2 hours.

**Supported formats**: FLAC, MP3, MP4, OGG, WAV, WEBM (audio) · MKV, MP4, OGV, WEBM (video)

### Generate transcript from audio file.

Transcribe the Travelbug wine tour guide audio, then chain AI_SENTIMENT and AI_COMPLETE on the transcript in a single CTE.

The Python cell below loads the audio file downloaded earlier and renders an in-notebook audio player - listen to the recording before you transcribe it.

In [ ]:
import base64
import glob
from IPython.display import display, HTML

print("=== Travelbug Wine Review ===")

mp3_files = glob.glob('../audio/*.mp3')

if not mp3_files:
    print('No .mp3 files found in ../audio/ folder. Run the Download Multimodal Files cell first.')
else:
    html = "<h3>Available Sound Files</h3>"
    for filepath in mp3_files:
        filename = filepath.split('/')[-1]
        with open(filepath, 'rb') as f:
            audio_b64 = base64.b64encode(f.read()).decode('utf-8')
        html += (
            f'<div style="margin-bottom:16px;">'
            f'<strong>{filename}</strong><br>'
            f'<audio controls><source src="data:audio/mpeg;base64,{audio_b64}" type="audio/mpeg">'
            f'Your browser does not support audio.</audio>'
            f'</div>'
        )
    display(HTML(html))

In [ ]:
%%sql -r Generate_transcript_from_audio_file_sql

WITH transcriptions AS (
    SELECT TO_VARCHAR(AI_TRANSCRIBE(TO_FILE('@{{user}}_genai_db.resources.genai2day/source_files',
        'garrandarra_merlot_review.mp3'))) AS wine_review 
)
SELECT
    AI_SENTIMENT(wine_review, ['taste', 'value', 'food', 'characteristics']) AS wine_sentiment,
    AI_COMPLETE('claude-haiku-4-5', CONCAT('List the characteristics of the wine: ', wine_review)) AS wine_assessment
FROM transcriptions;

In [ ]:
# Format output from the previous cell
df = Generate_transcript_from_audio_file_sql.to_pandas()

# Get the assessment string from the DataFrame
generated_response = df["WINE_ASSESSMENT"].iloc[0]

# Replace the literal '\\n' string with a true newline character '\n'
corrected_text = generated_response.replace('\\n', '\n')

print(corrected_text)

## AI-Powered JOIN Operations

`AI_FILTER` classifies text or images as **TRUE** or **FALSE** based on a natural language condition - enabling semantic filtering in WHERE clauses and JOIN conditions that traditional exact-match SQL cannot express.

### Filter reviews using AI_FILTER.

Use AI_FILTER in a WHERE clause to keep only bookings where the review is classified as positive and recommending the activity.

### Adventure reviews classification using AI_FILTER.

Classify free-text reviews as positive using AI_FILTER - a semantic filter rather than keyword matching.

The query joins activity, booking, and review tables, applies a minimum length filter, then uses AI_FILTER with a PROMPT to return only rows the model classifies as positive.

In [ ]:
%%sql -r Adventure_reviews_classification_sql
SELECT 
  -- STRUCTURED DATA: Basic business information
  a.name AS activity_name,
  a.location AS activity_location, 
  a.price AS base_price,
  b.total_price AS booking_price,
  
  -- UNSTRUCTURED DATA: Review content  
  r.review_text,
  r.review_date,
  
  -- AI ANALYSIS: Identify positive reviews using AI_FILTER + PROMPT
  AI_FILTER(
    PROMPT('This review is positive and recommends the activity: {0}', r.review_text)
  ) AS is_positive_review

FROM {{user}}_genai_db.transformed.activity a

-- TRADITIONAL JOINS: Connect structured tables using foreign keys
INNER JOIN {{user}}_genai_db.transformed.booking b ON a.activity_id = b.activity_id
INNER JOIN {{user}}_genai_db.transformed.review r ON b.booking_id = r.booking_id

-- BASIC FILTERS: Only activities with meaningful reviews  
WHERE r.review_text IS NOT NULL
  AND LENGTH(r.review_text) > 20
  -- AI-POWERED FILTER: Only show activities with positive reviews
  AND is_positive_Review

ORDER BY b.booking_date DESC
LIMIT 8;

### Semantic JOIN across data types.

Instead of joining on a shared key, use AI_FILTER as the JOIN condition - the model evaluates whether each review mentions a given activity, linking the tables by meaning rather than by ID.

In [ ]:
%%sql -r Advanced_semantic_join_operations_sql
SELECT 
    -- Select the customer's review text and alias it as "CUSTOMER FEEDBACK"
    r.review_text AS "CUSTOMER FEEDBACK",
    -- Select the name of the activity that the AI matched to the review
    a.name AS "MATCHING ACTIVITY",
    -- Include the full description of the matched activity
    a.description,
    -- Select the date the review was posted
    r.review_date
FROM
    -- Start with the customer reviews table
    {{user}}_genai_db.transformed.review r
-- Use a LEFT JOIN to attempt to match each review with an activity
LEFT JOIN
    {{user}}_genai_db.transformed.activity a
-- The JOIN condition is powered by an AI function instead of traditional keys (like review.activity_id = activity.id)
ON AI_FILTER(
    -- Use a PROMPT to ask the AI a true/false question for every possible review/activity pair
    PROMPT('You are provided customer feedback and an activity description. Check if this activity is in the 
    customer feedback. Customer feedback: {0}; \n\nActivity: {1}', 
            r.review_text, a.description) -- Pass the review text ({0}) and activity description ({1}) to the prompt
) = TRUE -- The join is made only if the AI determines the activity is a good match for the feedback
WHERE r.review_text IS NOT NULL -- Filter out any empty reviews
    AND LENGTH(r.review_text) > 25 -- Only consider reviews with more than 25 characters to ensure they have substance
ORDER BY r.review_date DESC -- Sort the results to show the most recent reviews first
LIMIT 10; -- Limit the output to the top 10 matches

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What capability does AI_COMPLETE have when used with multimodal models?", "options": ["A) It can only process text inputs", "B) AI_COMPLETE can analyze images when using multimodal models like Claude", "C) It requires a separate image processing service", "D) It converts images to text before processing"], "hash": "2c6a3f77fc4251c4d5f77ed5a57dbba8"},
    {"q": "What is the primary function of AI_TRANSCRIBE?", "options": ["A) It translates audio from one language to another", "B) It generates audio from text", "C) AI_TRANSCRIBE converts audio files into text transcripts", "D) It enhances audio quality"], "hash": "1bb642121a9b94096460c91517614942"},
    {"q": "How does AI_FILTER differ from traditional SQL filtering?", "options": ["A) AI_FILTER is faster than WHERE clauses", "B) AI_FILTER only works with numeric data", "C) AI_FILTER requires pre-defined filter categories", "D) AI_FILTER classifies data based on semantic meaning rather than exact matching"], "hash": "3369920c9ad9adf74c21778a445b3209"},
    {"q": "What makes multimodal AI models unique?", "options": ["A) Multimodal models can process both text and images in the same query", "B) They only work with structured data", "C) They require separate API calls for each data type", "D) They are limited to English language content"], "hash": "6f0fbe3a59cdb9243b5ec61623d0f9d2"},
    {"q": "What type of operations does AI_FILTER enable in SQL queries?", "options": ["A) Only exact string matching operations", "B) AI_FILTER enables semantic JOIN operations across different data types", "C) Only numeric comparisons", "D) Only date-based filtering"], "hash": "bc610bde82a5427d9004e35a3633d827"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ **AI_COMPLETE is Multimodal**: AI_COMPLETE works with text, images, and video - using vision-capable models like `claude-4-haiku` and `gemini-3.1-pro` to analyze visual content, classify emotions, compare images, and assess risks.

❄️ **Image Analysis with PROMPT() and TO_FILE()**: Use `PROMPT('instruction {0}', TO_FILE(...))` to pass images into AI_COMPLETE. Use `{0}`, `{1}` placeholders to compare multiple images in a single call.

❄️ **Video Analysis for Structured Metadata Extraction**: Video files can be analyzed with AI_COMPLETE to extract structured JSON metadata — identifying products, settings, and details directly from footage. Store the video as a FILE column in a table, define a JSON schema for the output, and query it with a vision-capable model.

❄️ **AI_TRANSCRIBE for Audio and Video**: AI_TRANSCRIBE converts speech from audio (MP3, WAV, FLAC) and video (MP4) files to text. Chain it with AI_SENTIMENT and AI_COMPLETE in CTEs for end-to-end analysis.

❄️ **AI_FILTER for Semantic SQL**: AI_FILTER classifies text as TRUE/FALSE based on natural language conditions - enabling semantic WHERE clauses and JOIN conditions that connect data by meaning rather than exact key matching.